In [1]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[0]  # go up from notebooks/
sys.path.append(str(ROOT))

In [2]:
from config import SEED, N_LIST, METHODS, RESULTS_PATH
from models.cvae import train_cvae_on_arrays, sample_cvae_dataset
from models.bootstrap import sample_bootstrap
from models.gmm import sample_gmm
import numpy as np
import pandas as pd
from metrics import *
import ipywidgets as widgets
from IPython.display import display

# Loading Data
1. Breast cancer
2. Pima Indians Diabetes
3. Somethng else 2

#### 1. Breast Cancer

| Dataset | Category | n | p | Class 0 | Class 1 | Source | Notes |
|---------|----------|---|---|--------|--------|--------|-------|
| Breast Cancer | clinical_tabular | 569 | 30 | 212 | 357 | sklearn | moderate n, moderate p, numeric only |

#### 2. Diabetes

| Dataset | Category | n | p | Class 0 | Class 1 | Source | Notes |
|---------|----------|---|---|--------|--------|--------|-------|
| Diabetes| metabolic_tabular | 768 | 8 | 500 | 268 | openml | high n, low p, numeric only |

In [3]:
from loaders import load_breast, load_diabetes
data = load_diabetes()

X = data["X"]
y = data["y"]
feature_names = data["feature_names"]

print(data["dataset"], data["category"])
print(X.shape, y.shape)
print(np.bincount(y))

diabetes metabolic_tabular
(768, 8) (768,)
[500 268]


# Helper Functions
- Stratify Sampler

In [4]:
X_small, y_small, idx_small = stratified_subsample(X, y, n0=23, n1=68, seed=42)
print(X_small.shape)
print(np.bincount(y_small))

(91, 8)
[23 68]


- Train & sample CVAE

In [5]:

best_state = train_cvae_on_arrays(
    X,
    y,
    seed=42,
    z_dim=16,
    hidden=128,
    beta=0.5,
    lr=1e-3,
    epochs=200,
    batch_size=32,
)

X_syn, y_syn = sample_cvae_dataset(
    best_state,
    n0=23,
    n1=68,
    seed=42
)

print(X_syn.shape)
print(np.bincount(y_syn))

Epoch  50 | val loss=4.6144 recon=2.4438 kl=4.3412
Epoch 100 | val loss=4.2760 recon=2.0567 kl=4.4386
Epoch 150 | val loss=4.4208 recon=2.1465 kl=4.5486
Epoch 200 | val loss=4.2890 recon=1.9778 kl=4.6222
(91, 8)
[23 68]


- bootstrap

In [6]:
X_syn_b, y_syn_b = sample_bootstrap(X_small, y_small, n0=23, n1=68, seed=42)

- gmm

In [7]:
X_syn_g, y_syn_g = sample_gmm(X_small, y_small, n0=23, n1=68, seed=42, n_components=2)

## Selection Side
- `forward` means keep top k
- `reverse` means drop top k

In [8]:
def select_feature_subset(X, feature_names, mode="full", ranked_idx=None, k=None, drop_idx=None):
    p = X.shape[1]
    
    if mode == "full":
        keep = np.arange(p)

    elif mode == "drop_one":
        keep = np.array([j for j in range(p) if j != drop_idx])

    elif mode == "forward":
        keep = np.array(ranked_idx[:k])

    elif mode == "reverse":
        keep = np.array(ranked_idx[k:])

    else:
        raise ValueError(f"Unknown mode: {mode}")

    return keep

In [9]:
def rank_features_by_rf_importance(X, y, seed=42, n_estimators=15):
    rf = RandomForestClassifier(n_estimators=n_estimators, random_state=seed)
    rf.fit(X, y)
    return np.argsort(rf.feature_importances_)[::-1]

# The actual tests with numbers or something

In [10]:
metrics = evaluate_all(X_real=X, X_syn=X_syn, y_real= y, y_syn = y_syn)
print(metrics)

RF trial 1/1
{'rf_auc_mean': np.float64(0.8444444444444444), 'rf_auc_sd': np.float64(0.0), 'rf_sep_mean': np.float64(0.8444444444444444), 'rf_sep_sd': np.float64(0.0), 'corr_mean_abs_diff': np.float64(0.09751815517370294), 'corr_max_abs_diff': np.float64(0.2620355239893517), 'pval_mean': np.float64(0.052808848151139294), 'pval_median': np.float64(0.020176472120781032), 'prop_significant': np.float64(0.75)}


### Results row

In [65]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

def run_experiment(n0, n1, feature_mode = "full", k = None, drop_idx = None, rows=[]):
    datasets = [
        load_breast,
        # load_diabetes,
    ]

    methods = ["bootstrap", "gmm", "cvae"]
    

    for load_fn in datasets:
        data = load_fn()
        X = data["X"]
        y = data["y"]
        feature_names = data["feature_names"]
        X_small, y_small, _ = stratified_subsample(
            X, y,
            n0=n0,
            n1=n1,
            seed=42
        )
        ranked_idx = rank_features_by_rf_importance(X_small, y_small, seed=42)
        keep = select_feature_subset(
            X_small,
            feature_names,
            mode=feature_mode,
            ranked_idx=ranked_idx,
            k=k,
            drop_idx=drop_idx
        )

        X_use = X_small[:, keep]
        feature_names_use = [feature_names[j] for j in keep]

        for method in methods:
            if method == "bootstrap":
                X_syn, y_syn = sample_bootstrap(X_use, y_small, n0, n1, seed=42)

            elif method == "gmm":
                X_syn, y_syn = sample_gmm(X_use, y_small, n0, n1, seed=42)

            elif method == "cvae":
                print("Training CVAE for", data["dataset"])
                best = train_cvae_on_arrays(
                    X_use,
                    y_small,
                    seed=42,
                    epochs=20,   
                    batch_size=32,

                )
                X_syn, y_syn = sample_cvae_dataset(best, n0, n1, seed=42)
            print(f"Evaluating {data['dataset']} | {method}")
            metrics = evaluate_all(X_real=X_use, y_real= y_small,  X_syn=X_syn, y_syn = y_syn)
            if feature_mode == "full":
                n_features_used_label = f"{X_use.shape[1]}"
            elif feature_mode == "drop_one":
                n_features_used_label = f"{X_use.shape[1]}"
            elif feature_mode == "forward":
                n_features_used_label = f"top {k}"
            elif feature_mode == "reverse":
                n_features_used_label = f"drop top {k}"
            else:
                n_features_used_label = "unknown"

            row = {
                "dataset": data["dataset"],
                "category": data["category"],
                "n0": n0,
                "n1": n1,
                "method": method,
                "feature_mode": feature_mode,
                "k": None,
                "drop_idx": None,
                "n_features_used": n_features_used_label,
                "n_features_total": X.shape[1],
                "rf_auc": metrics["rf_auc_mean"],
                "corr_mean_abs_diff": metrics["corr_mean_abs_diff"],
                "prop_significant": metrics["prop_significant"],
            }

            if feature_mode in {"forward", "reverse"}:
                row["k"] = k
            elif feature_mode == "drop_one":
                row["drop_idx"] = drop_idx

            rows.append(row)

    # df = pd.DataFrame(rows)
    # display(df)

In [74]:
rows = []
run_experiment(n0=60, n1= 60,  feature_mode = "full", rows = rows)
df = pd.DataFrame(rows)
display(df)

Evaluating breast_cancer | bootstrap
RF trial 1/1
Evaluating breast_cancer | gmm
RF trial 1/1
Training CVAE for breast_cancer
Evaluating breast_cancer | cvae
RF trial 1/1


,dataset,category,n0,n1,method,feature_mode,k,drop_idx,n_features_used,n_features_total,rf_auc,corr_mean_abs_diff,prop_significant
0,breast_cancer,clinical_tabular,60,60,bootstrap,full,None,None,30,30,0.295556,0.061283,0.000000
1,breast_cancer,clinical_tabular,60,60,gmm,full,None,None,30,30,0.635556,0.051594,0.000000
2,breast_cancer,clinical_tabular,60,60,cvae,full,None,None,30,30,0.964444,0.396633,0.266667


# Full Data Run
does synthetic quality depend on sample size?

In [68]:
rows = []

for n0, n1 in [(23,68),(35,35),(50,50),(60,60)]:
    run_experiment(
        n0,
        n1,
        feature_mode="full",
        rows=rows
    )

df = pd.DataFrame(rows)
display(df)

Evaluating breast_cancer | bootstrap
RF trial 1/1
Evaluating breast_cancer | gmm
RF trial 1/1
Training CVAE for breast_cancer
Evaluating breast_cancer | cvae
RF trial 1/1
Evaluating breast_cancer | bootstrap
RF trial 1/1
Evaluating breast_cancer | gmm
RF trial 1/1
Training CVAE for breast_cancer
Evaluating breast_cancer | cvae
RF trial 1/1
Evaluating breast_cancer | bootstrap
RF trial 1/1
Evaluating breast_cancer | gmm
RF trial 1/1
Training CVAE for breast_cancer
Evaluating breast_cancer | cvae
RF trial 1/1
Evaluating breast_cancer | bootstrap
RF trial 1/1
Evaluating breast_cancer | gmm
RF trial 1/1
Training CVAE for breast_cancer
Evaluating breast_cancer | cvae
RF trial 1/1


,dataset,category,n0,n1,method,feature_mode,k,drop_idx,n_features_used,n_features_total,rf_auc,corr_mean_abs_diff,prop_significant
0,breast_cancer,clinical_tabular,23,68,bootstrap,full,None,None,30,30,0.415556,0.042124,0.000000
1,breast_cancer,clinical_tabular,23,68,gmm,full,None,None,30,30,0.728889,0.050148,0.000000
2,breast_cancer,clinical_tabular,23,68,cvae,full,None,None,30,30,1.000000,0.352032,0.400000
3,breast_cancer,clinical_tabular,35,35,bootstrap,full,None,None,30,30,0.384444,0.074395,0.000000
4,breast_cancer,clinical_tabular,35,35,gmm,full,None,None,30,30,0.413333,0.058211,0.000000
5,breast_cancer,clinical_tabular,35,35,cvae,full,None,None,30,30,1.000000,0.343472,0.233333
6,breast_cancer,clinical_tabular,50,50,bootstrap,full,None,None,30,30,0.293333,0.064342,0.000000
7,breast_cancer,clinical_tabular,50,50,gmm,full,None,None,30,30,0.644444,0.054187,0.033333
8,breast_cancer,clinical_tabular,50,50,cvae,full,None,None,30,30,1.000000,0.248185,0.200000
9,breast_cancer,clinical_tabular,60,60,bootstrap,full,None,None,30,30,0.295556,0.061283,0.000000


# Drop One Test
Does one feature control the realism?

In [77]:
rows = []

for j in range(30):
    run_experiment(60,60,feature_mode="drop_one",drop_idx=j, rows = rows)

df = pd.DataFrame(rows)
display(df)

Evaluating breast_cancer | bootstrap
RF trial 1/1
Evaluating breast_cancer | gmm
RF trial 1/1
Training CVAE for breast_cancer
Evaluating breast_cancer | cvae
RF trial 1/1
Evaluating breast_cancer | bootstrap
RF trial 1/1
Evaluating breast_cancer | gmm
RF trial 1/1
Training CVAE for breast_cancer
Evaluating breast_cancer | cvae
RF trial 1/1
Evaluating breast_cancer | bootstrap
RF trial 1/1
Evaluating breast_cancer | gmm
RF trial 1/1
Training CVAE for breast_cancer
Evaluating breast_cancer | cvae
RF trial 1/1
Evaluating breast_cancer | bootstrap
RF trial 1/1
Evaluating breast_cancer | gmm
RF trial 1/1
Training CVAE for breast_cancer
Evaluating breast_cancer | cvae
RF trial 1/1
Evaluating breast_cancer | bootstrap
RF trial 1/1
Evaluating breast_cancer | gmm
RF trial 1/1
Training CVAE for breast_cancer
Evaluating breast_cancer | cvae
RF trial 1/1
Evaluating breast_cancer | bootstrap
RF trial 1/1
Evaluating breast_cancer | gmm
RF trial 1/1
Training CVAE for breast_cancer
Evaluating breast_c

,dataset,category,n0,n1,method,feature_mode,k,drop_idx,n_features_used,n_features_total,rf_auc,corr_mean_abs_diff,prop_significant
0,breast_cancer,clinical_tabular,60,60,bootstrap,drop_one,None,0,29,30,0.268889,0.060927,0.000000
1,breast_cancer,clinical_tabular,60,60,gmm,drop_one,None,0,29,30,0.371111,0.048061,0.000000
2,breast_cancer,clinical_tabular,60,60,cvae,drop_one,None,0,29,30,1.000000,0.302848,0.275862
3,breast_cancer,clinical_tabular,60,60,bootstrap,drop_one,None,1,29,30,0.228889,0.061763,0.000000
4,breast_cancer,clinical_tabular,60,60,gmm,drop_one,None,1,29,30,0.391111,0.062012,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,breast_cancer,clinical_tabular,60,60,gmm,drop_one,None,28,29,30,0.464444,0.064749,0.103448
86,breast_cancer,clinical_tabular,60,60,cvae,drop_one,None,28,29,30,1.000000,0.296827,0.310345
87,breast_cancer,clinical_tabular,60,60,bootstrap,drop_one,None,29,29,30,0.255556,0.062490,0.000000
88,breast_cancer,clinical_tabular,60,60,gmm,drop_one,None,29,29,30,0.471111,0.067811,0.137931


In [ ]:
rows = []
for k in [3,5,10,15]:
    run_experiment(60,60,feature_mode="forward",k=k, rows = rows)
df = pd.DataFrame(rows)
display(df)

Evaluating breast_cancer | bootstrap
RF trial 1/1
Evaluating breast_cancer | gmm
RF trial 1/1
Training CVAE for breast_cancer
Evaluating breast_cancer | cvae
RF trial 1/1
Evaluating breast_cancer | bootstrap
RF trial 1/1
Evaluating breast_cancer | gmm
RF trial 1/1
Training CVAE for breast_cancer
Evaluating breast_cancer | cvae
RF trial 1/1
Evaluating breast_cancer | bootstrap
RF trial 1/1
Evaluating breast_cancer | gmm
RF trial 1/1
Training CVAE for breast_cancer
Evaluating breast_cancer | cvae
RF trial 1/1
Evaluating breast_cancer | bootstrap
RF trial 1/1
Evaluating breast_cancer | gmm
RF trial 1/1
Training CVAE for breast_cancer
Evaluating breast_cancer | cvae
RF trial 1/1


,dataset,category,n0,n1,method,feature_mode,k,drop_idx,n_features_used,n_features_total,rf_auc,corr_mean_abs_diff,prop_significant
0,breast_cancer,clinical_tabular,60,60,bootstrap,drop_one,NaN,0.0,29,30,0.268889,0.060927,0.000000
1,breast_cancer,clinical_tabular,60,60,gmm,drop_one,NaN,0.0,29,30,0.371111,0.048061,0.000000
2,breast_cancer,clinical_tabular,60,60,cvae,drop_one,NaN,0.0,29,30,1.000000,0.302848,0.275862
3,breast_cancer,clinical_tabular,60,60,bootstrap,drop_one,NaN,1.0,29,30,0.228889,0.061763,0.000000
4,breast_cancer,clinical_tabular,60,60,gmm,drop_one,NaN,1.0,29,30,0.391111,0.062012,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,breast_cancer,clinical_tabular,60,60,gmm,forward,10.0,NaN,top 10,30,0.435556,0.019180,0.000000
98,breast_cancer,clinical_tabular,60,60,cvae,forward,10.0,NaN,top 10,30,0.944444,0.188219,0.300000
99,breast_cancer,clinical_tabular,60,60,bootstrap,forward,15.0,NaN,top 15,30,0.317778,0.043472,0.000000
100,breast_cancer,clinical_tabular,60,60,gmm,forward,15.0,NaN,top 15,30,0.482222,0.029610,0.000000
